# 第 77 课：流式推理、幂等重试、取消与背压

上一课解决 SLO、尾延迟和准入控制；这一课处理服务已经开始**逐 token 输出**时可能发生的事：客户端断线、重试、取消、超时、过慢消费与 KV 容量耗尽。

**本课成果：** 一个完全离线的单进程推理状态机，使用真正的微型因果 Transformer、两个 Q/V LoRA 版本和真实 CPU K/V 页张量；并在这些会话上生成、回放、确认及终止流式事件。

它不是生产 HTTP/SSE 服务器，不包含进程崩溃后的持久幂等、网络分布式事务、线程安全锁或 GPU 融合 PagedAttention；计时单位是**逻辑 tick**，不是毫秒。生成事件承载 token ID，真实文本产品还需要增量 UTF-8 解码。

## 使用方法与边界

从**全新内核顺序运行全部单元**；默认会运行 107 项固定检查和一份完整场景演练。Notebook 自包含所有函数与类，不依赖配套 `.py` 或上一课文件。

只要求现有 PyTorch。没有下载模型、网络请求、外部数据或真实用户凭证。模型权重是课堂随机初始化的冻结小模型，无法用来宣称自然语言质量。

搭配 `.py` 可以独立使用：

```bash
python lesson_77_inference_stream_retry_idempotency.py --checks-only
python lesson_77_inference_stream_retry_idempotency.py --demo-only
python lesson_77_inference_stream_retry_idempotency.py
python lesson_77_inference_stream_retry_idempotency.py --reload-run lesson_77_outputs/实际目录
```

本文件中完整演练默认开启。练习题检查默认关闭，参考答案放在最后。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第77课：推理服务幂等重试、流式续传、取消与背压。

独立离线PyTorch教学实验；真实微型因果Transformer/LoRA增量KV张量
与完整块前缀缓存；模拟确定性单进程事件循环的流式服务可靠性。
不建立HTTP服务器、不保证进程崩溃后的持久幂等、不声称生产并发。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import tempfile
from collections import Counter, OrderedDict
import secrets
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；本课不自动安装或下载模型。") from exc

VERSION = "lesson77-stream-retry-v1"
SOURCES = {
    "pagedattention": "https://arxiv.org/abs/2309.06180",
    "transformers_cache": "https://huggingface.co/docs/transformers/main/en/cache_explanation",
    "pytorch_attention": "https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html",
    "pytorch_inference": "https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html",
    "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
    "vllm_prefix_cache": "https://docs.vllm.ai/en/latest/design/prefix_caching/",
    "vllm_security": "https://docs.vllm.ai/en/stable/usage/security/",
    "vllm_internals": "https://vllm.ai/blog/2025-09-05-anatomy-of-vllm",
    "vllm_metrics": "https://github.com/vllm-project/vllm/blob/main/docs/design/metrics.md",
    "vllm_benchmark": "https://docs.vllm.ai/en/latest/benchmarking/cli/",
    "vllm_request_metrics": "https://docs.vllm.ai/en/latest/features/per_request_metrics/",
    "triton_batcher": "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html",
    "sre_overload": "https://sre.google/sre-book/handling-overload/",
    "sse": "https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events",
    "sse_standard": "https://html.spec.whatwg.org/multipage/server-sent-events.html",
    "aws_idempotency": "https://aws.amazon.com/builders-library/making-retries-safe-with-idempotent-APIs/",
    "ietf_idempotency_draft": "https://datatracker.ietf.org/doc/html/draft-ietf-httpapi-idempotency-key-header-08",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
print('PyTorch:',torch.__version__)
print('CUDA available:',torch.cuda.is_available())
print('逻辑时间、单线程CPU；无真实网络服务。')

PyTorch: 2.10.0+cpu
CUDA available: False
逻辑时间、单线程CPU；无真实网络服务。


## 1. 为什么需要真实 KV 状态？

如果只是把一个 JSON 状态从 `running` 改成 `cancelled`，无法证明取消以后物理 K/V 已经释放。本课沿用真实 CPU 张量组成的页池：

```text
KV backing [physical_page, layer, K/V, slot, head, head_dim]
   ↑ 会话block_table、引用计数、空闲页与cache pin
```

模型逐 token 计算新的 K/V、写入页，再读取已有页计算因果注意力。**每次活跃会话会核对增量 logits 和同一模型全上下文前向输出是否一致。**

分词器是明确标注的教学 byte-merge，并非 Hugging Face FastTokenizer；聊天控制符与普通用户正文分开编码。真实模型应换用其官方 tokenizer 和模板。

In [3]:
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [4]:
def build_models(seed: int=7201):
    """同一随机基座 + 两份独立Q/V LoRA。版本只是不可变教学对象。"""
    require(type(seed) is int and seed >= 0, 'seed非法')
    seed_all(seed)
    tok = ByteMergeChatTokenizer()
    cfg = ToyConfig(tok.vocab_size, dim=32, heads=4, layers=2, ff_dim=64, max_length=192)
    base = TinyChatLM(cfg)
    a = inject_lora(base, rank=4, alpha=8.0)
    b = inject_lora(base, rank=4, alpha=8.0)
    # B=0时LoRA作用为零；通过不同adapter B制造第二个可核对的版本。
    generator = torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for _,param in adapters(b):
            if param is not None and param.ndim == 2 and param.shape[1] == 4:  # B:[out,rank]
                param.copy_(torch.randn(param.shape, generator=generator)*0.12)
    models={'v0001':a.requires_grad_(False).eval(),
            'v0002':b.requires_grad_(False).eval()}
    require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
            '服务模型必须全部冻结')
    require(base_hash(models['v0001']) == base_hash(models['v0002']),
            '两个版本必须来自同一基座')
    require(state_hash(models['v0001'].state_dict()) != state_hash(models['v0002'].state_dict()),
            '版本权重应该不同')
    return tok,models

In [5]:
tok_demo,models_demo=build_models(7701)
print('教学词表:',tok_demo.vocab_size)
print('模型版本:',list(models_demo))
print('同一个冻结基座:',base_hash(models_demo['v0001'])==base_hash(models_demo['v0002']))
print('不同adapter权重:',state_hash(models_demo['v0001'].state_dict())!=state_hash(models_demo['v0002'].state_dict()))

教学词表: 281
模型版本: ['v0001', 'v0002']
同一个冻结基座: True
不同adapter权重: True


### KV 页分配、事务与回滚

一个 token 只有全部层都成功写入 K/V，才提交到会话序列。若页容量不足，保持已提交内容不变，不静默覆盖缓存。释放时清零物理页。

前缀缓存可以有自己的 pin，所以**结束活跃会话**不等于**全池已完全无占用**；最终还需驱逐 idle cache pin 后清零核对。

In [6]:
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u

In [7]:
class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

In [8]:
# 直接打开一个真实KV会话；比较完整前向，并释放所有物理页。
probe_pool=PagedKVPool(24,4,next(iter(models_demo.values())).cfg,set(models_demo))
probe_runner=CachedModelRunner(tok_demo,models_demo,probe_pool)
probe_ids=tok_demo.apply_chat_template([{'role':'user','content':'hello'}],add_generation_prompt=True)
probe_runner.open_prefill('probe','v0001',probe_ids)
print('prefill token数:',len(probe_ids))
print('KV与full logits最大差异:',probe_runner.assert_consistent('probe'))
print('物理占用块:',probe_pool.usage()['live_blocks'])
probe_runner.close('probe')
print('close后活跃KV块:',probe_pool.audit()['live_blocks'])

prefill token数: 7
KV与full logits最大差异: 2.980232238769531e-07
物理占用块: 2
close后活跃KV块: 0


## 2. 完整块前缀缓存为什么需要身份边界？

相同文本并不够：模型权重、LoRA、聊天模板、可信共享范围与媒体身份都必须匹配，才能复用已计算的 KV。只缓存**完整已提交块**，命中后续算时仍验证 logits 精度。

租户/主体在教学类中来自可信服务器侧身份映射，**不能直接把 HTTP 客户端自报的 `principal` 当鉴权结果**。

本地示例的随机共享盐只是区分缓存范围，不构成真实认证或密钥管理。

In [9]:
@dataclass(frozen=True)
class CacheRequest:
    """教学内部调用：principal必须由可信网关鉴权后传入，不能让客户端自行声明。"""
    sid: str
    principal: str
    version: str
    messages: tuple[tuple[str,str], ...]
    template_revision: str = "chat-v1"
    media_bytes: bytes | None = None
    add_generation_prompt: bool = True


class TrustedScopeAuthority:
    """服务器控制隔离组与加密随机盐，不接受请求自行提供salt。

    本地示例没有HTTP/登录鉴权。真实调用者必须由网关先确认principal身份，
    本类不防御攻击者能直接调用内部Python对象的情形。
    """
    def __init__(self, principal_groups: dict[str,str]):
        require(type(principal_groups) is dict and bool(principal_groups), '需要非空可信身份组')
        require(all(type(k) is str and k and type(v) is str and v
                    for k,v in principal_groups.items()),'可信身份/共享组非法')
        self._groups=principal_groups.copy()
        self._secrets={g:secrets.token_bytes(32) for g in set(principal_groups.values())}

    def group(self,principal: str) -> str:
        require(type(principal) is str and principal in self._groups,
                '未知或未认证主体不能使用Prefix Cache')
        return self._groups[principal]

    def fingerprint(self,principal: str) -> str:
        group=self.group(principal)
        return hashlib.sha256(self._secrets[group]).hexdigest()

    def rotate(self,group: str) -> str:
        require(group in self._secrets, '未知共享组')
        old=hashlib.sha256(self._secrets[group]).hexdigest()
        self._secrets[group]=secrets.token_bytes(32)
        require(self.fingerprint(next(k for k,v in self._groups.items() if v==group))!=old,
                '加密随机盐轮换未生效')
        return old

    def shared(self,p1:str,p2:str)->bool:
        return self.group(p1)==self.group(p2)


@dataclass(frozen=True)
class CacheIdentity:
    version: str
    model_hash: str
    base_hash: str
    tokenizer_hash: str
    template_hash: str
    scope_hash: str
    media_hash: str
    compute_hash: str


def request_tokens(tok:ByteMergeChatTokenizer,req:CacheRequest)->list[int]:
    require(isinstance(req,CacheRequest),'必须传入CacheRequest')
    require(type(req.messages) is tuple and bool(req.messages),'消息必须为非空tuple')
    require(type(req.add_generation_prompt) is bool,'生成提示选项非法')
    bounded_str(req.template_revision,64,'模板修订号')
    require(bool(req.template_revision),'模板修订号不能为空')
    messages=[]
    for item in req.messages:
        require(type(item) is tuple and len(item)==2 and item[0] in tok.role_ids
                and type(item[1]) is str,'消息必须为(role,content)')
        messages.append({'role':item[0],'content':item[1]})
    return tok.apply_chat_template(messages,add_generation_prompt=req.add_generation_prompt)


def make_identity(req:CacheRequest,authority:TrustedScopeAuthority,tok:ByteMergeChatTokenizer,
                  models:dict[str,TinyChatLM],saved_hashes:dict[str,str]) -> CacheIdentity:
    require(req.version in models and req.version in saved_hashes,'版本没有被可信模型库登记')
    authority.group(req.principal)
    require(type(req.media_bytes) in (bytes,type(None)),'媒体身份必须取自受信预处理的原始字节')
    require(req.media_bytes is None or len(req.media_bytes)<=4096,
            '教学元数据媒体上限4096字节；不允许超大请求键')
    # 禁止在同一个版本名称下原地修改模型而继续读取旧KV。
    model=models[req.version]
    current=state_hash(model.state_dict())
    require(current==saved_hashes[req.version], '热更新必须显式替换并失效缓存，不能原地修改模型')
    return CacheIdentity(
        version=req.version,
        model_hash=current,
        base_hash=base_hash(model),
        tokenizer_hash=digest(tok.contract()),
        template_hash=digest({'family':'toy-message-boundary','revision':req.template_revision,
                              'generation_prompt':req.add_generation_prompt,
                              'role_end':'assistant=eos,others=im_end',
                              'tokenizer':tok.contract()}),
        scope_hash=authority.fingerprint(req.principal),
        media_hash=('none' if req.media_bytes is None else
                    hashlib.sha256(req.media_bytes).hexdigest()),
        compute_hash=digest({'cfg':asdict(model.cfg),'dtype':str(next(model.parameters()).dtype),
                             'position':'absolute','attention':'full causal',
                             'kv_layout':'blocks,layer,kv,slot,heads,dim',
                             'runtime':'toy-cpu-fp32-v1'}),
    )


def prefix_keys(identity:CacheIdentity,ids:list[int],block_size:int)->list[str]:
    require(isinstance(identity,CacheIdentity),'缺少可信缓存身份')
    require(type(block_size) is int and block_size>0,'block_size必须为正整数')
    require(type(ids) is list and all(type(t) is int and t>=0 for t in ids),'token列表非法')
    parent=digest({'kind':'scoped-prefix-root','version':VERSION,'identity':asdict(identity)})
    keys=[]
    for start in range(0,len(ids)//block_size*block_size,block_size):
        block=ids[start:start+block_size]
        parent=digest({'kind':'full-prefix-block','parent_hash':parent,
                       'position':start,'token_ids':block})
        keys.append(parent)
    return keys


@dataclass
class CachedBlock:
    key: str
    physical_page: int
    identity: CacheIdentity
    prefix_tokens: tuple[int,...]   # 仅在可信进程内保留，用于哈希碰撞检测；报告禁止展示
    block_tokens: tuple[int,...]
    parent_key: str | None
    end_logits: torch.Tensor
    kv_hash: str
    logits_hash: str
    last_access_tick: int

In [10]:
class PrefixKVPool(PagedKVPool):
    """缓存pin与活动会话引用共同拥有物理页。只发布已提交的完整块。

    与工业vLLM不同：CPU同步、预分配Tensor、无真实异步锁、无融合页注意力核。
    精确前缀tokens用于碰撞检测，存储成本未计入KV字节预算。
    """
    def __init__(self,capacity_blocks:int,block_size:int,model_cfg:ToyConfig,versions:set[str],
                 cache_capacity_blocks:int|None=None):
        super().__init__(capacity_blocks,block_size,model_cfg,versions)
        cache_capacity_blocks=capacity_blocks if cache_capacity_blocks is None else cache_capacity_blocks
        require(type(cache_capacity_blocks) is int and 0<=cache_capacity_blocks<=capacity_blocks,
                '缓存块上限不能超过物理容量')
        self.cache_capacity_blocks=cache_capacity_blocks
        self.cache:dict[str,CachedBlock]={}
        self.clock=0
        self.hit_blocks=0;self.hit_tokens=0;self.miss_requests=0
        self.promotions=0;self.evictions=0;self.invalidations=0
        self.skipped_promotions=0;self.zero_hit_requests=0

    def _tick(self):
        self.clock+=1
        return self.clock

    def _page_hash(self,bid:int)->str:
        return state_hash({'physical_KV':self.data[bid]})

    def _validate_entry(self,e:CachedBlock,identity:CacheIdentity,desired_prefix:tuple[int,...],
                        requested_key:str):
        require(e.key==requested_key and e.identity==identity and
                e.prefix_tokens==desired_prefix,'哈希碰撞或缓存身份/前缀不一致，必须拒绝命中')
        require(e.block_tokens==desired_prefix[-self.block_size:]
                and len(e.block_tokens)==self.block_size,'只能命中完整块')
        require(e.kv_hash==self._page_hash(e.physical_page),'缓存KV字节损坏，拒绝复用')
        require(e.logits_hash==state_hash({'end_logits':e.end_logits}),
                '缓存块末端logits字节损坏，拒绝复用')
        require(self.refs[e.physical_page]>=1,'缓存页已经释放，索引悬空')

    def lookup(self,identity:CacheIdentity,ids:list[int])->list[CachedBlock]:
        keys=prefix_keys(identity,ids,self.block_size)
        found=[]
        for i,key in enumerate(keys):
            e=self.cache.get(key)
            if e is None:break
            self._validate_entry(e,identity,tuple(ids[:(i+1)*self.block_size]),key)
            # 顺序访问同时更新LRU；匹配只返回连续前缀，不能跳过中间缺失块。
            e.last_access_tick=self._tick()
            found.append(e)
        return found

    def _evict_one_idle(self)->bool:
        choices=[e for e in self.cache.values() if self.refs[e.physical_page]==1]
        if not choices:return False
        victim=min(choices,key=lambda e:(e.last_access_tick,e.key))
        del self.cache[victim.key]
        self._decref(victim.physical_page)
        self.evictions+=1
        return True

    def _alloc(self):
        if not self.free:
            self._evict_one_idle()
        return super()._alloc()

    def begin(self,sid,token_id,max_length):
        # 父类在发现free为空时会先报OOM；尝试驱逐idle cache才进入它的begin。
        if sid in self.sessions:
            s=self.sessions[sid]
            if s.pending is None and not self.free:
                slot=len(s.tokens)%self.block_size
                needs_page=slot==0 or (slot>0 and self.refs[s.block_table[-1]]>1)
                if needs_page:self._evict_one_idle()
        return super().begin(sid,token_id,max_length)

    def publish(self,sid:str,index:int,key:str,identity:CacheIdentity,
                prefix_tokens:tuple[int,...],end_logits:torch.Tensor,
                parent_key:str|None)->bool:
        require(sid in self.sessions,'发布会话不存在')
        s=self.sessions[sid]
        require(s.pending is None and s.version==identity.version,'不允许发布未提交/跨版本KV')
        require(type(index) is int and 0<=index<len(s.block_table),'发布块下标错误')
        require(len(prefix_tokens)==(index+1)*self.block_size and
                tuple(s.tokens[:len(prefix_tokens)])==prefix_tokens,'前缀token身份错误')
        require(end_logits.shape==(self.cfg.vocab_size,) and end_logits.dtype==self.dtype and
                bool(torch.isfinite(end_logits).all()),'末端logits形状/数值错误')
        chain=prefix_keys(identity,list(prefix_tokens),self.block_size)
        require(chain and chain[-1]==key and (chain[-2] if len(chain)>1 else None)==parent_key,
                '发布哈希链与token前缀不一致')
        if key in self.cache:
            self._validate_entry(self.cache[key],identity,prefix_tokens,key)
            return False
        if self.cache_capacity_blocks==0:
            self.skipped_promotions+=1
            return False
        if len(self.cache)>=self.cache_capacity_blocks and not self._evict_one_idle():
            self.skipped_promotions+=1
            return False
        bid=s.block_table[index]
        require(self.refs[bid]>=1,'发布物理页没有会话所有权')
        self.refs[bid]+=1             # 独立cache pin：请求结束后仍可命中
        logit=end_logits.detach().cpu().contiguous().clone()
        self.cache[key]=CachedBlock(key=key,physical_page=bid,identity=identity,
             prefix_tokens=prefix_tokens,block_tokens=prefix_tokens[-self.block_size:],
             parent_key=parent_key,end_logits=logit,kv_hash=self._page_hash(bid),
             logits_hash=state_hash({'end_logits':logit}),last_access_tick=self._tick())
        self.promotions+=1
        self.audit()
        return True

    def invalidate(self,predicate:Callable[[CachedBlock],bool]|None=None)->int:
        """移除pin。即使仍有活动会话，只有它们的引用会保留KV。"""
        selected=[key for key,e in self.cache.items() if predicate is None or predicate(e)]
        for key in selected:
            e=self.cache.pop(key)
            self._decref(e.physical_page)
        self.invalidations+=len(selected)
        self.audit()
        return len(selected)

    def advance_clock_and_expire(self,ticks:int,max_idle_ticks:int)->int:
        require(type(ticks) is int and ticks>=0 and type(max_idle_ticks) is int and max_idle_ticks>=0,
                '逻辑时间参数非法')
        self.clock+=ticks
        return self.invalidate(lambda e: self.clock-e.last_access_tick>max_idle_ticks)

    def cache_usage(self)->dict:
        total=self.usage()
        idle_cached=sum(self.refs[e.physical_page]==1 for e in self.cache.values())
        return {'cache_blocks':len(self.cache),'cache_capacity_blocks':self.cache_capacity_blocks,
                'idle_cached_blocks':idle_cached,'active_cached_blocks':len(self.cache)-idle_cached,
                'cache_pins':len(self.cache),'cache_held_bytes':len(self.cache)*self.bytes_per_block,
                'hit_blocks':self.hit_blocks,'hit_tokens':self.hit_tokens,
                'promotions':self.promotions,'evictions':self.evictions,
                'invalidations':self.invalidations,'skipped_promotions':self.skipped_promotions,
                'live_blocks':total['live_blocks'],'free_blocks':total['free_blocks'],
                'session_count':len(self.sessions),'physical_storage_bytes':total['physical_storage_bytes']}

    def usage(self)->dict:
        result=super().usage()
        result['cache_pins']=len(self.cache)
        result['cache_held_bytes']=len(self.cache)*self.bytes_per_block
        # 物理页可以只由缓存持有，因此旧课的logical_refs-live会变负：需把pin计入。
        result['saved_physical_blocks_by_sharing']=(result['logical_page_references']+
                                                     len(self.cache)-result['live_blocks'])
        return result

    def audit(self)->dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话模型版本不在白名单')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '会话页数与token长度不匹配')
            require(len(s.block_table)==len(set(s.block_table)),'同一会话重复引用页')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'页ID越界')
                expected[bid]+=1
        cache_pages=set()
        for key,e in self.cache.items():
            require(key==e.key and e.physical_page not in cache_pages,'重复pin或缓存键/页错配')
            cache_pages.add(e.physical_page)
            require(0<=e.physical_page<self.capacity_blocks,'缓存物理页非法')
            require(len(e.prefix_tokens)%self.block_size==0 and len(e.block_tokens)==self.block_size,
                    '缓存只允许完整块')
            expected[e.physical_page]+=1
        require(expected==self.refs,'会话引用+缓存pin不等于实际物理引用数')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free-list不守恒')
        require(len(set(self.free))==len(self.free),'free-list有重复')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),'空闲页未清零')
        u=self.usage()
        require(u['saved_physical_blocks_by_sharing']>=0,'物理共享账本负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '字节账本不守恒')
        return u

In [11]:
class PrefixCacheRunner:
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],
                 pool:PrefixKVPool,authority:TrustedScopeAuthority):
        require(isinstance(pool,PrefixKVPool),'需要有缓存所有权的KV页池')
        self.tok=tok;self.models=models;self.pool=pool;self.authority=authority
        self.runner=CachedModelRunner(tok,models,pool)
        self.bound:dict[str,CacheIdentity]={}
        self.events:list[dict]=[]

    def admit(self,req:CacheRequest)->dict:
        require(isinstance(req,CacheRequest),'请求类型非法')
        self.pool._sid(req.sid)
        require(req.sid not in self.pool.sessions,'sid不可重复入场')
        # 注意：principal须来自已鉴权的内部调用上下文，不应接受HTTP客户端自报。
        ids=request_tokens(self.tok,req)
        require(bool(ids) and len(ids)<=self.pool.cfg.max_length,'提示过长或为空')
        ident=make_identity(req,self.authority,self.tok,self.models,self.runner.model_hashes)
        hits=self.pool.lookup(ident,ids)
        hit_count=len(hits)*self.pool.block_size
        keys=prefix_keys(ident,ids,self.pool.block_size)
        self.pool.open(req.sid,req.version)
        self.bound[req.sid]=ident
        end_logits:dict[int,torch.Tensor]={}
        try:
            state=self.pool.sessions[req.sid]
            for i,e in enumerate(hits):
                self.pool.refs[e.physical_page]+=1
                state.block_table.append(e.physical_page)
                state.tokens.extend(e.block_tokens)
                end_logits[i]=e.end_logits.detach().clone()
            if hits:
                self.runner.last_logits[req.sid]=hits[-1].end_logits.detach().clone()
            self.pool.audit()
            for t in ids[hit_count:]:
                out=self.runner.step(req.sid,t)
                n=len(self.pool.sessions[req.sid].tokens)
                if n%self.pool.block_size==0:
                    end_logits[n//self.pool.block_size-1]=out.detach().clone()
            require(self.pool.sessions[req.sid].tokens==ids,'前缀命中恢复后的token序列错误')
            require(req.sid in self.runner.last_logits,'请求没有任何合法预测logits')
            # 先计算整个prompt，再发布完整块；失败入场不残留不完整页索引。
            parent=None
            for i,key in enumerate(keys):
                prefix=tuple(ids[:(i+1)*self.pool.block_size])
                require(i in end_logits,'完整块末端logits没有计算/回填')
                self.pool.publish(req.sid,i,key,ident,prefix,end_logits[i],parent)
                parent=key
            self.pool.hit_blocks+=len(hits);self.pool.hit_tokens+=hit_count
            if not hits:self.pool.zero_hit_requests+=1
            max_delta=self.runner.assert_consistent(req.sid)
            result={'sid':req.sid,'version':req.version,'prompt_tokens':len(ids),
                    'hit_blocks':len(hits),'hit_tokens':hit_count,'new_tokens_computed':len(ids)-hit_count,
                    'max_logits_abs_delta':max_delta,'cache_blocks_after':len(self.pool.cache),
                    'scope':'internal_authenticated_principal'}
            self.events.append(result.copy())
            return result
        except Exception:
            # 失败不留下活动会话/泄漏物理页；不回滚之前已发生的idle LRU驱逐。
            if req.sid in self.pool.sessions:
                self.runner.close(req.sid)
            self.bound.pop(req.sid,None)
            self.pool.audit()
            raise

    def continue_token(self,sid:str,token_id:int)->float:
        require(sid in self.bound and sid in self.pool.sessions,'不存在的会话')
        identity=self.bound[sid]
        require(identity.model_hash==state_hash(self.models[identity.version].state_dict()),
                '活跃会话期间模型权重不可原地热修改')
        self.runner.step(sid,token_id)
        return self.runner.assert_consistent(sid)

    def fork_session(self,parent:str,child:str,principal:str):
        require(parent in self.bound and parent in self.pool.sessions,'父会话不存在')
        require(child not in self.bound and child not in self.pool.sessions,'子会话已经存在')
        identity=self.bound[parent]
        require(self.authority.fingerprint(principal)==identity.scope_hash,
                'fork只允许相同可信缓存共享范围，不能跨租户')
        self.runner.fork(parent,child,identity.version)
        self.bound[child]=identity
        self.pool.audit()

    def close(self,sid:str):
        require(sid in self.bound,'未授权/已经关闭的会话')
        self.runner.close(sid)
        self.bound.pop(sid)
        self.pool.audit()

    def invalidate_version(self,version:str)->int:
        require(version in self.models,'未知版本')
        return self.pool.invalidate(lambda e:e.identity.version==version)

    def rotate_group(self,group:str)->int:
        old_fp=self.authority.rotate(group)
        return self.pool.invalidate(lambda e:e.identity.scope_hash==old_fp)

    def replace_version(self,version:str,new_model:TinyChatLM)->int:
        require(version in self.models,'未知版本不能替换')
        require(all(s.version!=version for s in self.pool.sessions.values()),
                '有该版本的活跃会话，禁止原地替换/撕裂读写')
        require(isinstance(new_model,TinyChatLM) and new_model.cfg==self.pool.cfg,
                '新模型配置或类型错误')
        require(next(new_model.parameters()).device.type=='cpu'
                and next(new_model.parameters()).dtype==torch.float32,
                '新模型必须是CPU FP32')
        require(all(not p.requires_grad for p in new_model.parameters()),'新模型必须全部冻结')
        replacement=copy.deepcopy(new_model).eval()
        invalidated=self.invalidate_version(version)
        self.models[version]=replacement
        self.runner.models[version]=replacement
        self.runner.model_hashes[version]=state_hash(replacement.state_dict())
        return invalidated

    def final_flush(self)->dict:
        require(not self.pool.sessions and not self.bound,'flush前必须结束全部会话')
        self.pool.invalidate()
        usage=self.pool.audit()
        require(usage['live_blocks']==0 and not bool(self.pool.data.count_nonzero()),
                '全部缓存清空后仍有KV残留')
        return usage

In [12]:
authority_demo=TrustedScopeAuthority({'alice':'group-A','bob':'group-B'})
cache_demo=PrefixKVPool(48,4,next(iter(models_demo.values())).cfg,set(models_demo),8)
prefix_demo=PrefixCacheRunner(tok_demo,models_demo,cache_demo,authority_demo)
for sid in ['prefix01','prefix02']:
    req=CacheRequest(sid=sid,principal='alice',version='v0001',messages=(('user','hello'),))
    result=prefix_demo.admit(req)
    print(sid,{'hit_tokens':result['hit_tokens'],'new_tokens_computed':result['new_tokens_computed'],
               'max_logits_abs_delta':result['max_logits_abs_delta']})
    prefix_demo.close(sid)
print('释放全部cache pins:',prefix_demo.final_flush()['live_blocks'])

prefix01 {'hit_tokens': 0, 'new_tokens_computed': 7, 'max_logits_abs_delta': 2.980232238769531e-07}
prefix02 {'hit_tokens': 4, 'new_tokens_computed': 3, 'max_logits_abs_delta': 2.980232238769531e-07}
释放全部cache pins: 0


## 3. 幂等键：标识意图，而不是标识TCP连接

客户端发送一个 **Idempotency-Key**。服务端按可信租户身份与 key 查找记录；第一次创建操作，重试只能获得**相同操作**的 ID，不重复创建模型会话。

必须核对请求指纹：同 key 却换了正文、模型版本、模板、生成预算或截止预算，应拒绝而不是静默覆盖。AWS 的可靠重试设计特别强调“同一个客户端请求ID，不同意图”必须检测。([AWS Builders Library](https://aws.amazon.com/builders-library/making-retries-safe-with-idempotent-APIs/))

HTTP Idempotency-Key 的 IETF 文档仍按所引用的 draft 版本阅读；本 Notebook 是**内部 Python 模拟协议**，不是声称已经实现该 HTTP 标准。([IETF draft](https://datatracker.ietf.org/doc/html/draft-ietf-httpapi-idempotency-key-header-08))

定义：

```text
幂等作用域 = (已认证主体, key)
请求指纹 = SHA256(请求的全部语义字段)
首次到达时 deadline_tick = 当前tick + timeout_ticks
重试不重置deadline
```

一旦取消或失败，仍保留 key 的墓碑；不能靠重试偷偷重启一次生成。

In [13]:
@dataclass(frozen=True)
class StreamRequest:
    """principal由可信网关注入，不可直接使用未鉴权HTTP字段。"""
    principal: str
    idempotency_key: str
    version: str
    text: str
    timeout_ticks: int = 12
    max_new_tokens: int = 5
    template_revision: str = "chat-v1"

    def validate(self):
        for name in ("principal", "idempotency_key", "version", "template_revision"):
            val=getattr(self,name)
            bounded_str(val,64,name)
            require(bool(re.fullmatch(r"[A-Za-z0-9_-]+",val)),name+"只允许安全字符")
        bounded_str(self.text,256,"text")
        require(0 < len(self.text.encode('utf-8')) <= 256,'空输入或超长输入')
        require(type(self.timeout_ticks) is int and 1 <= self.timeout_ticks <= 1000,'timeout_ticks非法')
        require(type(self.max_new_tokens) is int and 1 <= self.max_new_tokens <= 12,'max_new_tokens非法')

    def fingerprint(self):
        self.validate()
        return digest(asdict(self))


@dataclass(frozen=True)
class StreamConfig:
    capacity_blocks: int = 96
    block_size: int = 4
    cache_capacity_blocks: int = 12
    max_sessions: int = 4
    max_records: int = 80
    max_unacked: int = 3
    slow_consumer_ticks: int = 8
    retention_ticks: int = 12

    def validate(self):
        for key,value in asdict(self).items():
            require(type(value) is int and value > 0,key+'必须为正整数')
        require(self.cache_capacity_blocks <= self.capacity_blocks,'cache超过物理页容量')
        require(self.max_unacked >= 2,'至少允许accepted及一次delta')
        require(self.retention_ticks >= 2,'保留期过短')


@dataclass
class StreamRecord:
    sid: str
    request: StreamRequest|None
    fingerprint: str
    identity_hash: str
    accepted_tick: int
    deadline_tick: int
    state: str = 'active'
    event_log: list[dict] = field(default_factory=list)
    acked_seq: int = -1
    last_ack_tick: int = 0
    output_ids: list[int] = field(default_factory=list)
    stop_reason: str|None = None
    end_tick: int|None = None
    prefill: dict|None = None
    playback_expired: bool = False


class RejectedStream(LessonError):
    """有明确拒绝原因的服务契约错误。"""


class ReplayGone(RejectedStream):
    """幂等墓碑还在，但重放窗口已过；绝不默默重新生成。"""

In [14]:
class StreamGateway:
    """单进程同步状态机。每一tick推进确定性顺序，不提供网络/事务/线程安全保证。

    关键约定：先完成真实KV写入，再提交delta事件；每个幂等key唯一活跃计算。
    客户端可能重复收到同一事件，用event id去重；不承诺exactly-once delivery。
    accepted事件占一个未ack槽；terminal可越过背压确保可靠结束信号。
    """
    terminal_states={'done','cancelled','deadline_exceeded','slow_consumer','kv_oom','capacity_rejected'}
    valid_events={'accepted','delta','done','cancelled','deadline_exceeded',
                  'slow_consumer','kv_oom','capacity_rejected'}

    def __init__(self,tok,models,authority,config:StreamConfig=StreamConfig()):
        config.validate()
        require(type(models) is dict and set(models)=={'v0001','v0002'},'教学必须使用两个固定模型版本')
        self.tok=tok;self.models=models;self.authority=authority;self.config=config
        cfg=next(iter(models.values())).cfg
        self.pool=PrefixKVPool(config.capacity_blocks,config.block_size,cfg,set(models),
                               cache_capacity_blocks=config.cache_capacity_blocks)
        self.runner=PrefixCacheRunner(tok,models,self.pool,authority)
        self.now=0
        self.records:dict[tuple[str,str],StreamRecord]={}
        self.sid_index:dict[str,StreamRecord]={}
        self.admissions=0;self.duplicate_submissions=0;self.rejected_overload=0
        self.total_tokens=0;self.backpressure_skips=0;self.deadline_count=0
        self.cancel_count=0;self.expiration_count=0;self.slow_count=0
        self.failures=0;self.max_pending_observed=0
        self.audit_log:list[dict]=[]

    def _id_key(self,principal,key):
        require(type(principal) is str and type(key) is str,'租户/幂等键必须为字符串')
        bounded_str(principal,64,'principal');bounded_str(key,64,'idempotency_key')
        self.authority.group(principal)  # 来自服务端可信映射
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',key)),'幂等键非法')
        return principal,key

    def _record(self,principal,key):
        k=self._id_key(principal,key)
        require(k in self.records,'未找到幂等操作；不能凭sid读取其他租户')
        return self.records[k]

    def _emit(self,record:StreamRecord,kind:str,data:dict):
        require(kind in self.valid_events,'未知事件类型')
        require(type(data) is dict and all(type(k) is str for k in data),'事件内容非法')
        require(not record.playback_expired,'重放期已过不能继续追加')
        seq=len(record.event_log)
        event={'id':f'{record.sid}:{seq}','seq':seq,'type':kind,'tick':self.now,
               'data':copy.deepcopy(data)}
        record.event_log.append(event)
        self.max_pending_observed=max(self.max_pending_observed,seq-record.acked_seq)
        return event

    def _active(self):
        return [r for r in self.records.values() if r.state=='active']

    def submit(self,request:StreamRequest):
        require(isinstance(request,StreamRequest),'需要StreamRequest')
        request.validate()
        key=self._id_key(request.principal,request.idempotency_key)
        fingerprint=request.fingerprint()
        if key in self.records:
            old=self.records[key]
            require(old.fingerprint==fingerprint,'同一租户同一幂等键不能修改语义参数')
            self.duplicate_submissions+=1
            return {'status':'expired' if old.playback_expired else 'reused',
                    'sid':old.sid,'state':old.state,'deadline_tick':old.deadline_tick,
                    'new_compute':False}
        require(len(self.records)<self.config.max_records,'幂等墓碑容量耗尽；拒绝而不是静默逐出')
        require(request.version in self.models,'不支持的模型版本')
        # 固定长度预算：拒绝上下文越界，不在计算之后悄悄截断。
        cache_req=CacheRequest(sid='unused',principal=request.principal,version=request.version,
            messages=(('user',request.text),),template_revision=request.template_revision)
        tokens=request_tokens(self.tok,cache_req)
        require(len(tokens)+request.max_new_tokens<=self.pool.cfg.max_length,'prompt+生成预算超过上下文')
        sid=f'rs{len(self.records):04d}'
        record=StreamRecord(sid=sid,request=request,fingerprint=fingerprint,
            identity_hash=digest({'fp':fingerprint,'model':self.runner.runner.model_hashes[request.version],
                                  'tokenizer':self.tok.contract()}),accepted_tick=self.now,
            deadline_tick=self.now+request.timeout_ticks,last_ack_tick=self.now)
        # 先占有幂等位置，然后尝试真实KV预填充；拒绝也保留terminal墓碑。
        self.records[key]=record;self.sid_index[sid]=record
        if len(self._active())>self.config.max_sessions:
            self.rejected_overload+=1
            self._terminate(record,'capacity_rejected',{'reason':'concurrent_session_limit'})
            return {'status':'capacity_rejected','sid':sid,'state':record.state,
                    'deadline_tick':record.deadline_tick,'new_compute':False}
        internal=CacheRequest(sid=sid,principal=request.principal,version=request.version,
                messages=(('user',request.text),),template_revision=request.template_revision)
        try:
            admitted=self.runner.admit(internal)
        except NoKVSpace:
            self.rejected_overload+=1
            self._terminate(record,'capacity_rejected',{'reason':'KV_capacity'})
            return {'status':'capacity_rejected','sid':sid,'state':record.state,
                    'deadline_tick':record.deadline_tick,'new_compute':False}
        except Exception:
            # 编程错误不能伪装为正常终态；清除尚未入场的墓碑并重新抛出。
            self.records.pop(key,None);self.sid_index.pop(sid,None)
            raise
        record.prefill=admitted
        self.admissions+=1
        self._emit(record,'accepted',{'version':request.version,'max_new_tokens':request.max_new_tokens,
                                     'deadline_tick':record.deadline_tick})
        self.audit_log.append({'tick':self.now,'op':'admit','sid':sid,'version':request.version})
        return {'status':'created','sid':sid,'state':record.state,
                'deadline_tick':record.deadline_tick,'new_compute':True}

    def _terminate(self,record:StreamRecord,kind:str,data:dict):
        require(kind in self.terminal_states,'终态类型不合法')
        require(record.state=='active','只能从active进入唯一终态')
        record.state=kind;record.stop_reason=kind;record.end_tick=self.now
        # 终态先释放资源再提交唯一终态事件，避免发了done却没回收KV。
        if record.sid in self.runner.bound:
            self.runner.close(record.sid)
        self._emit(record,kind,{**data,'generated_tokens':len(record.output_ids)})
        self.audit_log.append({'tick':self.now,'op':kind,'sid':record.sid})
        if kind=='cancelled':self.cancel_count+=1
        if kind=='deadline_exceeded':self.deadline_count+=1
        if kind=='slow_consumer':self.slow_count+=1
        if kind=='kv_oom':self.failures+=1

    def cancel(self,principal,key):
        r=self._record(principal,key)
        if r.state=='active':
            self._terminate(r,'cancelled',{'reason':'explicit_request'})
            return {'status':'cancelled','cancelled_now':True}
        return {'status':r.state,'cancelled_now':False}

    def _generate_one(self,record:StreamRecord):
        require(record.state=='active','只能推进活跃请求')
        req=record.request;sid=record.sid
        logits=self.runner.runner.last_logits[sid]
        # 教学贪心生成；固定模型/版本、同样的logits决定token，不使用未声明采样温度。
        next_id=int(logits.argmax())
        try:
            self.runner.continue_token(sid,next_id)  # 先真实KV提交
        except NoKVSpace:
            self._terminate(record,'kv_oom',{'reason':'physical_KV_pool_full'})
            return
        record.output_ids.append(next_id)
        self.total_tokens+=1
        if next_id==self.tok.eos_token_id:
            self._terminate(record,'done',{'finish_reason':'eos'})
        else:
            self._emit(record,'delta',{'token_id':next_id,'position':len(record.output_ids)-1})
            if len(record.output_ids)>=req.max_new_tokens:
                self._terminate(record,'done',{'finish_reason':'length'})

    def tick(self,steps=1):
        require(type(steps) is int and 1<=steps<=1000,'tick步数非法')
        for _ in range(steps):
            self.now+=1
            for r in list(self._active()):
                if self.now>=r.deadline_tick:
                    self._terminate(r,'deadline_exceeded',{'reason':'absolute_request_deadline'})
                    continue
                pending=len(r.event_log)-1-r.acked_seq
                if pending>=self.config.max_unacked:
                    self.backpressure_skips+=1
                    if self.now-r.last_ack_tick>=self.config.slow_consumer_ticks:
                        self._terminate(r,'slow_consumer',{'reason':'unacked_window_timeout'})
                    continue
                self._generate_one(r)
            self.prune()
            self.audit()

    @staticmethod
    def render_sse(event):
        require(type(event) is dict and set(event)=={'id','seq','type','tick','data'},'事件结构非法')
        require(type(event['id']) is str and bool(re.fullmatch(r'rs\d{4}:\d+',event['id'])),
                'SSE id非法')
        require(event['type'] in StreamGateway.valid_events,'事件类型非法')
        # 所有data均以单行JSON编码，避免把用户输入作为SSE控制行。
        data=canonical({'tick':event['tick'],**event['data']})
        return f"id: {event['id']}\nevent: {event['type']}\ndata: {data}\n\n"

    @staticmethod
    def parse_sse_frame(frame):
        require(type(frame) is str and frame.endswith('\n\n'),'单帧SSE必须以空行结束')
        parts=frame[:-2].split('\n')
        require(len(parts)==3,'仅接受id/event/data三行教学帧')
        keys=['id: ','event: ','data: ']
        require(all(line.startswith(prefix) for line,prefix in zip(parts,keys)),'SSE字段或顺序非法')
        event_id,typ,raw=[line[len(prefix):] for line,prefix in zip(parts,keys)]
        require(bool(re.fullmatch(r'rs\d{4}:\d+',event_id)) and typ in StreamGateway.valid_events,
                'SSE id/type非法')
        payload=json.loads(raw)
        require(type(payload) is dict and type(payload.get('tick')) is int,'数据体非法')
        return {'id':event_id,'type':typ,'data':payload}

    def _cursor(self,record,last_event_id):
        if last_event_id is None:return -1
        bounded_str(last_event_id,64,'Last-Event-ID')
        require(last_event_id.startswith(record.sid+':'),'事件ID不属于本操作/本租户')
        tail=last_event_id[len(record.sid)+1:]
        require(bool(re.fullmatch(r'(0|[1-9]\d*)',tail)),'事件序号非法')
        cursor=int(tail)
        require(cursor<len(record.event_log),'客户端不能声称未来事件或已丢失事件')
        return cursor

    def poll(self,principal,key,last_event_id=None):
        r=self._record(principal,key)
        if r.playback_expired:raise ReplayGone('事件保留期已结束；墓碑仍阻止隐式重跑')
        cursor=self._cursor(r,last_event_id)
        return [copy.deepcopy(e) for e in r.event_log if e['seq']>cursor]

    def ack(self,principal,key,last_event_id):
        r=self._record(principal,key)
        if r.playback_expired:raise ReplayGone('事件已经过期')
        cursor=self._cursor(r,last_event_id)
        require(cursor>=r.acked_seq,'ack不能倒退')
        # 同序号重复ack为幂等操作，时间不刷新，避免无限延长慢客户端租约。
        changed=cursor>r.acked_seq
        if changed:
            r.acked_seq=cursor;r.last_ack_tick=self.now
        return {'acked_seq':r.acked_seq,'new_ack':changed,
                'pending':len(r.event_log)-1-r.acked_seq}

    def prune(self):
        for r in self.records.values():
            if r.state!='active' and not r.playback_expired and r.end_tick is not None:
                if self.now-r.end_tick>=self.config.retention_ticks:
                    r.event_log=[];r.output_ids=[];r.prefill=None;r.request=None
                    r.playback_expired=True;self.expiration_count+=1

    def audit(self):
        usage=self.pool.audit()
        active=self._active()
        require(len(active)<=self.config.max_sessions,'活动会话超过准入上限')
        require(len(self.records)<=self.config.max_records,'墓碑记录上限不守恒')
        for r in self.records.values():
            if not r.playback_expired:
                require(r.request is not None and r.fingerprint==r.request.fingerprint(),
                        '操作指纹与请求内容改变')
            if r.state=='active':
                require(r.sid in self.pool.sessions and r.sid in self.runner.bound,
                        '活跃记录缺KV会话/runner绑定')
                require(not r.playback_expired and r.end_tick is None,'活动记录不能过期或结束')
            else:
                require(r.state in self.terminal_states and r.end_tick is not None,'终态非法')
                require(r.sid not in self.pool.sessions and r.sid not in self.runner.bound,
                        '终态有KV会话泄漏')
            if r.playback_expired:
                require(not r.event_log,'过期后不能留下事件数据')
                continue
            require([e['seq'] for e in r.event_log]==list(range(len(r.event_log))),
                    '事件序列必须连续、不能重复提交')
            require(all(e['id']==f'{r.sid}:{e["seq"]}' for e in r.event_log),'事件ID与记录不匹配')
            terminals=[e for e in r.event_log if e['type'] in self.terminal_states]
            require(len(terminals)==(0 if r.state=='active' else 1),'必须恰好一次终态事件')
            require(r.state=='active' or r.event_log[-1]['type']==r.state,'终态必须最后发布')
            require(-1<=r.acked_seq<len(r.event_log),'ack不能越过已提交事件')
            for e in r.event_log:
                require(e['type'] in self.valid_events,'未知事件')
                if e['type']=='delta':
                    require(type(e['data'].get('token_id')) is int,'delta必须包含真实token ID')
            if r.state=='active':
                # accepted计入窗口；终态事件允许多占一个槽确保可见。
                require(len(r.event_log)-1-r.acked_seq<=self.config.max_unacked,
                        '活跃请求超过未确认消息上限')
        require(usage['session_count']==len(active),'活跃记录数与实际KV会话数不一致')
        return {'tick':self.now,'records':len(self.records),'active':len(active),
                'terminal':len(self.records)-len(active),'KV':self.pool.cache_usage(),
                'tokens_generated':self.total_tokens,'duplicate_submissions':self.duplicate_submissions,
                'backpressure_skips':self.backpressure_skips,'max_pending':self.max_pending_observed}

    def final_flush(self):
        for r in list(self._active()):
            self._terminate(r,'cancelled',{'reason':'server_shutdown'})
        usage=self.runner.final_flush()
        require(usage['live_blocks']==0 and usage['session_count']==0 and
                not bool(self.pool.data.count_nonzero()),'最终物理KV仍被持有')
        self.audit()
        return usage

In [15]:
def new_service(seed=7701,config:StreamConfig|None=None,model_bundle=None):
    torch.set_num_threads(1)
    tok,models=build_models(seed) if model_bundle is None else model_bundle
    authority=TrustedScopeAuthority({'alice':'tenant-A','bob':'tenant-B','alice2':'tenant-A'})
    return StreamGateway(tok,models,authority,config or StreamConfig())


def safe_request(key,text='hello',principal='alice',version='v0001',timeout=20,max_tokens=5):
    return StreamRequest(principal,key,version,text,timeout,max_tokens)

In [16]:
srv=new_service(7701)
req=safe_request('demo','hello',timeout=15,max_tokens=5)
a=srv.submit(req)
b=srv.submit(req)
print('首次提交:',a)
print('重试同sid、无新增计算:',b)
try:
    srv.submit(safe_request('demo','world',timeout=15,max_tokens=5))
except LessonError as error:
    print('同键不同意图被拒绝:',str(error))
print('当前真实KV会话:',srv.pool.usage()['session_count'])

首次提交: {'status': 'created', 'sid': 'rs0000', 'state': 'active', 'deadline_tick': 15, 'new_compute': True}
重试同sid、无新增计算: {'status': 'reused', 'sid': 'rs0000', 'state': 'active', 'deadline_tick': 15, 'new_compute': False}
同键不同意图被拒绝: 同一租户同一幂等键不能修改语义参数
当前真实KV会话: 1


## 4. 流式事件：先提交 KV，再发布 token

SSE 的帧通常用 `id`、`event`、`data` 表示，每帧以空行结束。客户端可使用最后一个收到的 ID 进行断线重连。([MDN](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events))

本课用一个更简单的应用协议：**每个 data 包含 JSON 形式的 token_id**，不是宣称可直接显示的 UTF-8 文本。token 可能来自 byte 片段，真实界面应安全地增量解码；不能把每个字节当作独立Unicode字符。

```text
id: rs0000:1
 event: delta
 data: {"token_id": ..., "position":...,"tick":...}

```

上面缩进只为了展示；实际 `render_sse()` 生成严格的无缩进三字段。用户正文不会直接拼进 `event:` 或 `id:` 控制行。

**顺序保证：** 成功的真实 KV 事务 → 保存 token 事件 → 可交付客户端。取消、超时和正常完成只发布一个终态，终态释放活跃 KV。

In [17]:
srv.tick(1)
first=srv.poll('alice','demo')
for event in first:
    print(srv.render_sse(event),end='')
print('增量full前向核对:',srv.runner.runner.assert_consistent(a['sid']))
print('客户端游标从0续传:',[x['id'] for x in srv.poll('alice','demo',first[0]['id'])])

id: rs0000:0
event: accepted
data: {"deadline_tick":15,"max_new_tokens":5,"tick":0,"version":"v0001"}

id: rs0000:1
event: delta
data: {"position":0,"tick":1,"token_id":242}

增量full前向核对: 3.5762786865234375e-07
客户端游标从0续传: ['rs0000:1']


## 5. 断线续传 ≠ 端到端 exactly-once delivery

服务器承诺：在该进程和幂等墓碑生命周期中，同 key 只有一份操作，事件 ID 对同一个操作稳定。客户端重连时可能重收已交付事件，必须按 ID 去重。

以下示范客户端既保留本地去重集合，又故意忘记游标：它会再次收到旧事件，却不会重复拼接回答。

`ack` 表示客户端确认已经处理某条事件；这个演示只有一个可信消费方，没有实现真实多消费者竞争、持久化 ack 或网络断线后的跨进程事务。不能把它当作端到端 exactly-once 证明。

In [18]:
class ReplayClient:
    """单消费方教学模型；真实多客户端ack需要单独的所有权与序列化策略。"""
    def __init__(self,principal,key):
        self.principal=principal;self.key=key
        self.seen:set[str]=set();self.ordered:list[dict]=[]
        self.cursor:str|None=None
        self.duplicates=0

    def receive(self,server:StreamGateway,*,ack=True,disconnect_after=None):
        events=server.poll(self.principal,self.key,self.cursor)
        for event in events:
            if event['id'] in self.seen:
                self.duplicates+=1
                continue
            # 模拟网络只收到前N条：cursor只在交付给本客户端后前进。
            if disconnect_after is not None and len(self.ordered)>=disconnect_after:
                break
            parsed=StreamGateway.parse_sse_frame(StreamGateway.render_sse(event))
            require(parsed['id']==event['id'] and parsed['type']==event['type'],'SSE帧往返不一致')
            self.seen.add(event['id']);self.ordered.append(event)
            self.cursor=event['id']
            if ack:server.ack(self.principal,self.key,self.cursor)
        return len(self.ordered)

    def restart_without_cursor(self):
        # 故意丢失游标但保留去重ID，模拟重复投递。
        self.cursor=None

    @property
    def tokens(self):
        return [e['data']['token_id'] for e in self.ordered if e['type']=='delta']

In [19]:
c=ReplayClient('alice','demo')
c.receive(srv,ack=True)
before=len(c.ordered)
c.restart_without_cursor()
c.receive(srv,ack=True)
print('客户端唯一事件:',len(c.ordered),'重复收到:',c.duplicates)
print('唯一token事件:',c.tokens)
print('服务器实际生成token:',srv._record('alice','demo').output_ids)
assert c.duplicates==before

客户端唯一事件: 2 重复收到: 2
唯一token事件: [242]
服务器实际生成token: [242]


## 6. 背压：慢客户端不能无限占用KV与内存

客户端一直不 ack 时，待确认事件窗口增长到 `max_unacked=3` 后，服务跳过后续解码，防止无限堆积。超过 `slow_consumer_ticks` 时进入明确终态并释放活跃KV。

若消费者后来确认，窗口恢复可继续解码；但 **ack 并不自动修改已生成 token 或模型版本**。

终态事件可以绕过已满窗口额外占一个槽，目的是让消费者有机会观察终止原因，所以终态流与活跃流的窗口上界不同。终态保留也受到 TTL 限制。

In [20]:
bp_srv=new_service(7710)
bp_req=safe_request('pause','world',timeout=30,max_tokens=8)
bp_srv.submit(bp_req)
bp_srv.tick(6)
bpr=bp_srv._record('alice','pause')
print('未ack窗口:',len(bpr.event_log)-1-bpr.acked_seq)
print('累计背压跳过:',bp_srv.backpressure_skips,'实际生成:',len(bpr.output_ids))
ReplayClient('alice','pause').receive(bp_srv,ack=True)
previous=len(bpr.output_ids)
bp_srv.tick(1)
print('ack后新增token:',len(bpr.output_ids)-previous)
bp_srv.final_flush()

未ack窗口: 3
累计背压跳过: 4 实际生成: 2
ack后新增token: 1


{'capacity_blocks': 96,
 'block_size': 4,
 'free_blocks': 96,
 'live_blocks': 0,
 'physical_storage_bytes': 196608,
 'bytes_per_token': 512,
 'bytes_per_block': 2048,
 'live_reserved_bytes': 0,
 'physical_used_slots': 0,
 'live_content_bytes': 0,
 'physical_internal_fragmentation_bytes': 0,
 'logical_token_uses': 0,
 'logical_page_references': 0,
 'saved_physical_blocks_by_sharing': 0,
 'session_count': 0,
 'peak_live_blocks': 3,
 'peak_sessions': 1,
 'allocation_count': 3,
 'cow_count': 0,
 'oom_count': 0,
 'zeroized_releases': 3,
 'cache_pins': 0,
 'cache_held_bytes': 0}

## 7. 取消、超时与容量拒绝：每一种终态都必须释放真实 KV

取消与断开网络连接不同：**断线只是失去接收者，取消是显式要求停止计算。**

超时预算来自初次入场的绝对逻辑截止 tick，重试不会续命；到截止 tick 时，先终止，后续不再生成 token。

容量不足时拒绝本次入场，并保留对应 key 的拒绝墓碑；重试同键不会创建另一次计算。OOM 回滚、取消与超时都要求先清理真实 KV，再发布唯一终态。

页池里 idle prefix cache pin 可能还保留，但活动会话不得继续占用。

In [21]:
lifecycle=new_service(7711)
x=lifecycle.submit(safe_request('cancel_me','hello',timeout=20,max_tokens=8))
print('取消:',lifecycle.cancel('alice','cancel_me'))
print('重复取消:',lifecycle.cancel('alice','cancel_me'))
print('活跃会话:',lifecycle.pool.usage()['session_count'])
y=lifecycle.submit(safe_request('expire_now','world',timeout=1,max_tokens=8))
lifecycle.tick(1)
print('截止后的状态:',lifecycle._record('alice','expire_now').state)
print('继续重试:',lifecycle.submit(safe_request('expire_now','world',timeout=1,max_tokens=8)))
print('释放全部物理页:',lifecycle.final_flush()['live_blocks'])

取消: {'status': 'cancelled', 'cancelled_now': True}
重复取消: {'status': 'cancelled', 'cancelled_now': False}
活跃会话: 0
截止后的状态: deadline_exceeded
继续重试: {'status': 'reused', 'sid': 'rs0001', 'state': 'deadline_exceeded', 'deadline_tick': 1, 'new_compute': False}
释放全部物理页: 0


## 8. 重放 TTL 与幂等墓碑：不能过期后悄悄重跑

终态完成后，事件日志只保留一段逻辑时间。到期后：

- 删除旧事件、原始请求正文和生成 token 暂存；
- 保留最低限度的 fingerprint 与终态墓碑，防止本进程内旧 key 意外重启；
- 访问旧事件明确报 `ReplayGone`，而不是给出全新序列。

这仍不是生产级数据保留方案：墓碑记录数在本进程内有固定上限，**满了就拒绝新操作**，不会静默删除旧墓碑；进程重启后该内存账本不保留。真实系统需要持久化、原子操作记录、保留期策略、数据治理与跨节点一致性。([AWS 重试设计](https://aws.amazon.com/builders-library/making-retries-safe-with-idempotent-APIs/))

In [22]:
ttl=new_service(7712,StreamConfig(retention_ticks=3))
request=safe_request('expired','hello',timeout=20,max_tokens=3)
ttl.submit(request);ttl.cancel('alice','expired')
print('过期前可重放条数:',len(ttl.poll('alice','expired')))
ttl.tick(3)
print('重试结果:',ttl.submit(request))
try:
    ttl.poll('alice','expired')
except ReplayGone as error:
    print('过期后明确拒绝:',str(error))
print('原始请求已清理:',ttl._record('alice','expired').request is None)
ttl.final_flush()

过期前可重放条数: 2
重试结果: {'status': 'expired', 'sid': 'rs0000', 'state': 'cancelled', 'deadline_tick': 20, 'new_compute': False}
过期后明确拒绝: 事件保留期已结束；墓碑仍阻止隐式重跑
原始请求已清理: True


{'capacity_blocks': 96,
 'block_size': 4,
 'free_blocks': 96,
 'live_blocks': 0,
 'physical_storage_bytes': 196608,
 'bytes_per_token': 512,
 'bytes_per_block': 2048,
 'live_reserved_bytes': 0,
 'physical_used_slots': 0,
 'live_content_bytes': 0,
 'physical_internal_fragmentation_bytes': 0,
 'logical_token_uses': 0,
 'logical_page_references': 0,
 'saved_physical_blocks_by_sharing': 0,
 'session_count': 0,
 'peak_live_blocks': 2,
 'peak_sessions': 1,
 'allocation_count': 2,
 'cow_count': 0,
 'oom_count': 0,
 'zeroized_releases': 2,
 'cache_pins': 0,
 'cache_held_bytes': 0}

## 9. 固定场景演练：真实输出，不伪造时延数据

完整演练包含：同键重试、事件重投、按ID去重、背压恢复、租户/版本隔离、取消、绝对截止与重放期满。所有输出来自真实微型模型和真实KV页，**时间仍仅为逻辑 tick**。

报告只含计数和脱敏状态；模型权重、内部身份盐与用户原文不会进入场景报告。

In [23]:
def run_scenarios(seed=7701,model_bundle=None):
    svc=new_service(seed,model_bundle=model_bundle)
    scenes={}
    req=safe_request('stable','hello',timeout=30,max_tokens=6)
    one=svc.submit(req);copy1=svc.submit(req)
    require(one['sid']==copy1['sid'] and not copy1['new_compute'],'重试制造重复运算')
    before=svc.runner.runner.tokens_processed
    svc.tick(1)
    # 重试不重置截止时间，也不再生成额外token。
    retry=svc.submit(req)
    require(retry['deadline_tick']==one['deadline_tick'] and
            svc.runner.runner.tokens_processed==before+1,'重试改变训练时钟或重复计算')
    client=ReplayClient('alice','stable')
    client.receive(svc,ack=True)
    previous=len(client.ordered)
    client.restart_without_cursor()
    client.receive(svc,ack=True)
    scenes['retry_replay']={'same_sid':one['sid']==copy1['sid']==retry['sid'],
                            'duplicate_receipt_events':client.duplicates,
                            'unique_events':len(client.ordered),
                            'single_delta_after_one_tick':sum(e['type']=='delta' for e in client.ordered)==1,
                            'deadline_unchanged':retry['deadline_tick']==one['deadline_tick']}
    # 背压：消费者停止ack，服务不超过上限；恢复后可以继续。
    req2=safe_request('paused','world',timeout=35,max_tokens=7)
    svc.submit(req2)
    svc.tick(6)
    r2=svc._record('alice','paused')
    pending=len(r2.event_log)-1-r2.acked_seq
    require(pending<=svc.config.max_unacked and svc.backpressure_skips>0,'背压失效')
    bp_before=len(r2.output_ids)
    c2=ReplayClient('alice','paused');c2.receive(svc,ack=True)
    svc.tick(2)
    scenes['backpressure']={'pending_at_pause':pending,'tokens_at_pause':bp_before,
       'tokens_after_ack':len(r2.output_ids),'skipped_ticks':svc.backpressure_skips,
       'max_pending':svc.config.max_unacked}
    # 同一租户相同key不同payload拒绝；不同租户同key完全独立。
    try:svc.submit(safe_request('stable','world'))
    except LessonError:conflict=True
    else:conflict=False
    other=svc.submit(safe_request('stable','hello',principal='bob',version='v0002',timeout=30,max_tokens=6))
    require(conflict and other['sid']!=one['sid'],'幂等必须有租户隔离和请求指纹')
    try:svc.poll('bob','stable',f'{one["sid"]}:0')
    except LessonError:cross_id=True
    else:cross_id=False
    scenes['isolation']={'different_payload_conflict':conflict,'tenant_scoped_key':other['sid']!=one['sid'],
                         'wrong_event_id_rejected':cross_id}
    # 取消：幂等重试返回相同终态，不重新运行，也不能取消其他操作。
    rc=svc.submit(safe_request('kill','answer',timeout=40,max_tokens=9))
    first_cancel=svc.cancel('alice','kill');second_cancel=svc.cancel('alice','kill')
    require(first_cancel['cancelled_now'] and not second_cancel['cancelled_now'],'取消幂等失败')
    rr=svc.submit(safe_request('kill','answer',timeout=40,max_tokens=9))
    require(rr['sid']==rc['sid'] and rr['state']=='cancelled','取消后重试重新生成')
    scenes['cancel']={'first':first_cancel['status'],'second_cancelled_now':second_cancel['cancelled_now'],
                       'same_sid':rr['sid']==rc['sid'],'KV_released':rc['sid'] not in svc.pool.sessions}
    # 超时：原始绝对截止时间，不允许通过重试延长。
    deadline=svc.submit(safe_request('deadline','42',timeout=1,max_tokens=8))
    svc.tick(1)
    late=svc.submit(safe_request('deadline','42',timeout=1,max_tokens=8))
    scenes['deadline']={'state':late['state'],'immutable_deadline':late['deadline_tick']==deadline['deadline_tick'],
                        'KV_released':deadline['sid'] not in svc.pool.sessions}
    # 物理KV页在活跃会话释放后，缓存pin可能仍在，最后显式flush。
    states={key:r.state for (_,key),r in svc.records.items()}
    for r in list(svc._active()):svc.cancel(r.request.principal,r.request.idempotency_key)
    terminal_usage=svc.audit()
    svc.tick(svc.config.retention_ticks)
    expired_retry=svc.submit(req)
    try:svc.poll('alice','stable')
    except ReplayGone:gone=True
    else:gone=False
    require(expired_retry['status']=='expired' and gone,'重放过期后不应隐式重跑')
    scenes['retention']={'retry_status':expired_retry['status'],'replay_gone':gone,
                          'records_tombstoned':len(svc.records)}
    final=svc.final_flush()
    scenes['resource']={'peak_live_blocks':final['peak_live_blocks'],
                        'allocated_pages':final['allocation_count'],
                        'zeroized_releases':final['zeroized_releases'],
                        'kv_live_after_flush':final['live_blocks'],
                        'prompt_tokens_processed':svc.runner.runner.tokens_processed,
                        'generation_tokens':svc.total_tokens,
                        'physical_bytes_reserved':final['physical_storage_bytes'],
                        'note':'CPU真实物理KV张量，tick不是毫秒，前缀缓存pin在final_flush后清空'}
    scenes['terminal_states_before_cleanup']=states
    return scenes

In [24]:
scenario=run_scenarios(7701)
print(json.dumps(scenario,ensure_ascii=False,indent=2))

{
  "retry_replay": {
    "same_sid": true,
    "duplicate_receipt_events": 2,
    "unique_events": 2,
    "single_delta_after_one_tick": true,
    "deadline_unchanged": true
  },
  "backpressure": {
    "pending_at_pause": 3,
    "tokens_at_pause": 2,
    "tokens_after_ack": 4,
    "skipped_ticks": 9,
    "max_pending": 3
  },
  "isolation": {
    "different_payload_conflict": true,
    "tenant_scoped_key": true,
    "wrong_event_id_rejected": true
  },
  "cancel": {
    "first": "cancelled",
    "second_cancelled_now": false,
    "same_sid": true,
    "KV_released": true
  },
  "deadline": {
    "state": "deadline_exceeded",
    "immutable_deadline": true,
    "KV_released": true
  },
  "retention": {
    "retry_status": "expired",
    "replay_gone": true,
    "records_tombstoned": 5
  },
  "resource": {
    "peak_live_blocks": 9,
    "allocated_pages": 12,
    "zeroized_releases": 12,
    "kv_live_after_flush": 0,
    "prompt_tokens_processed": 45,
    "generation_tokens": 10,
    "

## 10. 自动检查与故意错误检测

下面的固定检查覆盖：特殊token、身份指纹、KV 精度、请求去重、租户隔离、帧结构、Last-Event-ID、ack、背压、慢客户端、绝对超时、容量/OOM、墓碑过期和物理清零。

**检查通过不是生产证明。** 本课是同步状态机，没有真实HTTP、多线程、分布式锁或掉电场景。尤其不能把单进程内的幂等操作误报为跨进程 exactly-once。

In [25]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def raises(self,name,fn,exceptions=(LessonError,)):
        try:fn()
        except exceptions:self.names.append(name);return
        raise AssertionError(name+': 应拒绝但没有拒绝')
    def close(self,name,a,b,atol=2e-5):
        self.ok(name,torch.allclose(torch.as_tensor(a,dtype=torch.float64),
                                   torch.as_tensor(b,dtype=torch.float64),atol=atol,rtol=2e-5))


def run_checks():
    torch.set_num_threads(1)
    c=Checks()
    tok,models=build_models(7701)
    c.ok('模型版本固定且不同',set(models)=={'v0001','v0002'} and
         state_hash(models['v0001'].state_dict())!=state_hash(models['v0002'].state_dict()))
    c.ok('两版本基座相同',base_hash(models['v0001'])==base_hash(models['v0002']))
    c.ok('所有模型参数冻结',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    c.ok('tokenizer有独立角色控制token',tok.role_ids['user']!=tok.role_ids['assistant'])
    c.ok('特殊控制符与正文分开',tok.encode('<eos>') != [tok.eos_token_id])
    c.raises('空请求拒绝',lambda:safe_request('a','').validate())
    c.raises('超长正文拒绝',lambda:safe_request('a','a'*257).validate())
    c.raises('ID换行注入拒绝',lambda:safe_request('a\nX','hello').validate())
    c.raises('非法版本键拒绝',lambda:safe_request('a','hello',version='v/001').validate())
    c.raises('负超时拒绝',lambda:StreamRequest('alice','a','v0001','hello',timeout_ticks=-1).validate())
    c.raises('零token预算拒绝',lambda:StreamRequest('alice','a','v0001','hello',max_new_tokens=0).validate())
    c.ok('超时属于幂等语义指纹',safe_request('k',timeout=1).fingerprint()!=safe_request('k',timeout=2).fingerprint())
    c.ok('版本属于幂等语义指纹',safe_request('k',version='v0001').fingerprint()!=safe_request('k',version='v0002').fingerprint())
    c.ok('正文属于幂等语义指纹',safe_request('k','hello').fingerprint()!=safe_request('k','world').fingerprint())
    c.raises('无效并发上限拒绝',lambda:StreamConfig(max_sessions=0).validate())
    c.raises('缓存容量大于物理容量拒绝',lambda:StreamConfig(capacity_blocks=4,cache_capacity_blocks=5).validate())
    c.raises('过小窗口拒绝',lambda:StreamConfig(max_unacked=1).validate())

    svc=new_service(7701)
    require(svc.authority.shared('alice','alice2'),'同组身份测试夹具错误')
    r=safe_request('stable','hello',timeout=30,max_tokens=6)
    original=svc.submit(r);r0=svc._record('alice','stable')
    c.ok('首次请求唯一物理会话',svc.pool.usage()['session_count']==1)
    c.ok('首次accepted事件序号0',len(r0.event_log)==1 and r0.event_log[0]['seq']==0)
    c.ok('accepted包含截止时间',r0.event_log[0]['data']['deadline_tick']==30)
    c.ok('首次accepted不含用户原文',not any('hello' in canonical(e) for e in r0.event_log))
    hash_before=state_hash(models['v0001'].state_dict())
    before=svc.runner.runner.tokens_processed
    again=svc.submit(r)
    c.ok('重复入场不新增KV会话',svc.pool.usage()['session_count']==1)
    c.ok('重复入场不重复prefill',svc.runner.runner.tokens_processed==before)
    c.ok('重复入场共用sid',again['sid']==original['sid'])
    c.ok('重复入场没有计算标记',not again['new_compute'])
    c.ok('同键截止时刻不可重置',again['deadline_tick']==original['deadline_tick'])
    c.ok('模型冻结未被prefill改动',state_hash(models['v0001'].state_dict())==hash_before)
    c.raises('同键不同正文拒绝',lambda:svc.submit(safe_request('stable','world',timeout=30,max_tokens=6)))
    c.raises('同键不同超时拒绝',lambda:svc.submit(safe_request('stable','hello',timeout=31,max_tokens=6)))
    c.raises('同键不同版本拒绝',lambda:svc.submit(safe_request('stable','hello',version='v0002',timeout=30,max_tokens=6)))
    c.raises('未认证主体拒绝',lambda:svc.submit(safe_request('stable','hello',principal='unknown')))
    svc.tick(1)
    c.ok('本tick只计算一个输出token',len(r0.output_ids)==1)
    delta=[e for e in r0.event_log if e['type']=='delta']
    c.ok('输出delta来自真实模型',len(delta)==1 and delta[0]['data']['token_id']==r0.output_ids[0])
    c.ok('增量KV与完整前向一致',svc.runner.runner.assert_consistent(r0.sid)<2e-5)
    frame=svc.render_sse(delta[0]);parsed=svc.parse_sse_frame(frame)
    c.ok('SSE三字段可回读',parsed['id']==delta[0]['id'] and parsed['type']=='delta')
    c.ok('SSE消息末尾空行',frame.endswith('\n\n'))
    c.ok('SSE JSON单行不注入结构行',frame.count('\ndata: ')==1 and frame.count('\nevent: ')==1)
    c.raises('SSE伪造多行拒绝',lambda:svc.parse_sse_frame(frame+'data: evil\n\n'))
    c.raises('SSE字段顺序错误拒绝',lambda:svc.parse_sse_frame(frame.replace('id: ','foo: ',1)))
    c.raises('SSE不完整帧拒绝',lambda:svc.parse_sse_frame(frame.rstrip()))
    events=svc.poll('alice','stable')
    c.ok('首次重放含accepted和delta',[e['seq'] for e in events]==[0,1])
    c.ok('Last-Event-ID只返回之后事件',len(svc.poll('alice','stable',events[0]['id']))==1)
    c.raises('未来事件编号拒绝',lambda:svc.poll('alice','stable',f'{r0.sid}:99'))
    c.raises('负数事件编号拒绝',lambda:svc.poll('alice','stable',f'{r0.sid}:-1'))
    c.raises('非规范带零编号拒绝',lambda:svc.poll('alice','stable',f'{r0.sid}:01'))
    c.raises('跨会话事件编号拒绝',lambda:svc.poll('alice','stable','rs9999:0'))
    c.raises('未来ack拒绝',lambda:svc.ack('alice','stable',f'{r0.sid}:99'))
    c.ok('已确认ID更新ack游标',svc.ack('alice','stable',events[0]['id'])['acked_seq']==0)
    last_ack_tick=r0.last_ack_tick
    c.ok('同一ack不刷新慢用户时钟',not svc.ack('alice','stable',events[0]['id'])['new_ack']
         and r0.last_ack_tick==last_ack_tick)
    c.raises('ack回退拒绝',lambda:svc.ack('alice','stable',f'{r0.sid}:0-'))
    client=ReplayClient('alice','stable')
    client.receive(svc,ack=True)
    count=len(client.ordered)
    client.restart_without_cursor();client.receive(svc,ack=True)
    c.ok('事件重复投递时客户端按ID去重',client.duplicates==count and len(client.ordered)==count)
    c.ok('实际token事件没有重复追加',client.tokens==[e['data']['token_id'] for e in delta])
    c.ok('重连不重新采样',svc._record('alice','stable').sid==original['sid'] and svc.admissions==1)

    # 多租户同字面key独立；跨租户事件ID不能复用。
    bob=svc.submit(safe_request('stable','hello',principal='bob',version='v0002',timeout=30,max_tokens=6))
    c.ok('不同租户同key获得不同sid',bob['sid']!=original['sid'])
    c.ok('不同租户同key独立预填充',svc.admissions==2)
    c.raises('跨租户事件ID不能用',lambda:svc.poll('bob','stable',f'{r0.sid}:0'))
    c.raises('未存在的幂等操作不能凭sid读',lambda:svc.poll('alice','not_a_key'))
    c.ok('不同版本KV身份不同',svc.runner.bound[original['sid']].model_hash!=svc.runner.bound[bob['sid']].model_hash)
    c.ok('同租户凭id仍必须使用对应key',svc._record('alice','stable').sid!=svc._record('bob','stable').sid)

    # 背压后旧会话继续占KV，确认后允许下一次解码。
    svc2=new_service(7702,StreamConfig(slow_consumer_ticks=20))
    svc2.submit(safe_request('bp','world',timeout=30,max_tokens=10))
    svc2.tick(7)
    br=svc2._record('alice','bp')
    c.ok('未ack事件不超过上限',len(br.event_log)-1-br.acked_seq==svc2.config.max_unacked)
    c.ok('停止解码时真实输出长度不变',len(br.output_ids)==svc2.config.max_unacked-1)
    c.ok('背压累计跳过tick',svc2.backpressure_skips>=4)
    old_tokens=len(br.output_ids)
    bp_client=ReplayClient('alice','bp');bp_client.receive(svc2,ack=True)
    svc2.tick(1)
    c.ok('确认窗口后解码恢复',len(br.output_ids)==old_tokens+1)
    c.ok('窗口内不会丢掉已提交事件',len(svc2.poll('alice','bp',None))>=svc2.config.max_unacked+1)
    c.ok('ack不足以自动完成推理',br.state=='active')
    svc2.cancel('alice','bp')
    c.ok('取消后释放会话资源',svc2.pool.usage()['session_count']==0)
    c.ok('取消后唯一terminal事件',sum(e['type'] in svc2.terminal_states for e in br.event_log)==1)
    c.ok('取消后不继续生成',not svc2.cancel('alice','bp')['cancelled_now'])
    c.ok('取消后重试不重新计算',svc2.submit(safe_request('bp','world',timeout=30,max_tokens=10))['state']=='cancelled')
    svc2.final_flush()

    # 慢客户端：总不ack时释放实际KV，终态允许绕过满窗口。
    slowcfg=StreamConfig(max_unacked=2,slow_consumer_ticks=3)
    slow=new_service(7703,slowcfg)
    slow.submit(safe_request('slow','hello',timeout=20,max_tokens=10))
    slow.tick(5)
    s=slow._record('alice','slow')
    c.ok('无ack触发slow_consumer',s.state=='slow_consumer')
    c.ok('慢客户端只发一次终态',sum(e['type']=='slow_consumer' for e in s.event_log)==1)
    c.ok('慢客户端终态即释放KV',s.sid not in slow.pool.sessions)
    c.ok('终态不被满窗口阻塞',len(s.event_log)-1-s.acked_seq==slowcfg.max_unacked+1)
    c.ok('慢客户端重复请求不会重启',slow.submit(safe_request('slow','hello',timeout=20,max_tokens=10))['state']=='slow_consumer')
    slow.final_flush()

    # 到达截止tick之前没有额外延长；超时与背压优先级明确。
    dead=new_service(7704)
    dreq=safe_request('dl','hello',timeout=1,max_tokens=9)
    d=dead.submit(dreq)
    dead.tick(1)
    dr=dead._record('alice','dl')
    c.ok('截止tick先超时，不额外生成',dr.state=='deadline_exceeded' and len(dr.output_ids)==0)
    c.ok('超时KV已释放',dr.sid not in dead.pool.sessions)
    c.ok('重试截止时刻固定',dead.submit(dreq)['deadline_tick']==d['deadline_tick'])
    c.ok('超时终态只有一个',sum(e['type']=='deadline_exceeded' for e in dr.event_log)==1)
    c.ok('超时发生后不能取消改写',not dead.cancel('alice','dl')['cancelled_now'] and dr.state=='deadline_exceeded')
    dead.final_flush()

    # 容量上限：第二个请求拒绝并形成幂等墓碑，第一会话保持。
    limited=new_service(7705,StreamConfig(max_sessions=1))
    yes=limited.submit(safe_request('one','hello',timeout=30))
    no=limited.submit(safe_request('two','world',timeout=30))
    c.ok('超过并发上限被拒绝',no['status']=='capacity_rejected')
    c.ok('拒绝后第一会话仍存活',limited._record('alice','one').state=='active')
    c.ok('拒绝不会增加会话',limited.pool.usage()['session_count']==1)
    c.ok('容量拒绝重试幂等',limited.submit(safe_request('two','world',timeout=30))['state']=='capacity_rejected')
    c.ok('拒绝有唯一终态',len(limited._record('alice','two').event_log)==1)
    limited.final_flush()

    # 很小的物理KV容量必须原子回滚并留下容量墓碑。
    micro=new_service(7706,StreamConfig(capacity_blocks=1,cache_capacity_blocks=1))
    mr=micro.submit(safe_request('big','hello',timeout=20))
    c.ok('真实KV不足明确容量拒绝',mr['status']=='capacity_rejected')
    c.ok('真实OOM不能残留半成品会话',micro.pool.usage()['session_count']==0)
    c.ok('真实OOM回滚并清零所有页',micro.pool.audit()['live_blocks']==0
         and not bool(micro.pool.data.count_nonzero()))
    c.ok('容量拒绝不能偷偷改变为EOS',micro._record('alice','big').state=='capacity_rejected')
    micro.final_flush()

    # 墓碑和数据保留：到期后既不泄漏明文，也不擅自重新运行。
    g=new_service(7707,StreamConfig(retention_ticks=3))
    req=safe_request('gone','hello',timeout=10)
    initial=g.submit(req)
    g.cancel('alice','gone')
    c.ok('取消终态一开始仍可重放',len(g.poll('alice','gone'))>=2)
    g.tick(3)
    old=g._record('alice','gone')
    c.ok('过期清理事件正文和原请求',old.playback_expired and not old.event_log
         and old.request is None and not old.output_ids)
    c.raises('过期游标应显式410语义',lambda:g.poll('alice','gone'),exceptions=(ReplayGone,))
    c.ok('过期重试返回expired墓碑',g.submit(req)['status']=='expired')
    c.ok('过期重试不再prefill',g.admissions==1)
    c.ok('已过期操作的sid不改变',g.submit(req)['sid']==initial['sid'])
    c.ok('过期后fingerprint仍可拒绝新意图',bool(old.fingerprint))
    c.raises('过期后同key不同意图仍拒绝',lambda:g.submit(safe_request('gone','world',timeout=10)))
    g.final_flush()
    few=new_service(7708,StreamConfig(max_records=1))
    few.submit(safe_request('key1','hello'));few.cancel('alice','key1');few.tick(20)
    c.raises('墓碑上限不能悄悄淘汰key',lambda:few.submit(safe_request('key2','hello')))
    few.final_flush()

    # 真实增量模型精度与缓存页复用另做直接核对。
    kv=new_service(7709)
    v1=kv.submit(safe_request('p1','hello',timeout=20))
    delta1=kv.runner.runner.assert_consistent(v1['sid'])
    kv.cancel('alice','p1')
    v2=kv.submit(safe_request('p2','hello',timeout=20))
    delta2=kv.runner.runner.assert_consistent(v2['sid'])
    c.ok('同身份重复prompt命中真实前缀块',kv._record('alice','p2').prefill['hit_tokens']>0)
    c.ok('KV命中后logits全前向精度一致',max(delta1,delta2)<2e-5)
    kv.cancel('alice','p2')
    final=kv.final_flush()
    c.ok('final_flush物理页完全回收',final['live_blocks']==0)
    c.ok('final_flush擦除所有KV字节',not bool(kv.pool.data.count_nonzero()))
    c.ok('真实物理KV有过非零分配',final['allocation_count']>0)
    c.ok('版本权重未变',state_hash(models['v0001'].state_dict())==hash_before)
    scenarios=run_scenarios(7701)
    c.ok('系统场景可复跑重试行为',scenarios['retry_replay']['same_sid'])
    c.ok('系统场景可复跑租户隔离',scenarios['isolation']['wrong_event_id_rejected'])
    c.ok('系统场景可复跑资源释放',scenarios['resource']['kv_live_after_flush']==0)
    c.ok('系统场景可复跑TTL失效',scenarios['retention']['replay_gone'])
    return {'passed':len(c.names),'names':c.names,
            'scope':'单进程CPU真实KV＋确定性tick；不测试真实HTTP网络/线程/重启恢复'}

In [26]:
checks=run_checks()
print('固定检查通过:',checks['passed'],'项')
print('范围:',checks['scope'])

固定检查通过: 107 项
范围: 单进程CPU真实KV＋确定性tick；不测试真实HTTP网络/线程/重启恢复


## 11. 三道独立练习（默认关闭）

**练习 A：请求语义指纹。** 给定 `StreamRequest`，返回 SHA256 的 64 字符十六进制摘要。相同请求应相同；文本、版本、超时或生成预算变化均必须改变。不要把到达 tick 作为指纹内容，也不要把原文直接当 key。

**练习 B：基于 Last-Event-ID 续传。** 返回指定 cursor 之后的事件，不重复已接收的事件；其他会话 ID 必须拒绝。真正HTTP服务还需按已授权身份验证。

**练习 C：单tick决策。** 截止 tick 优先于背压；窗口满就暂停，否则允许生成。此练习不模拟慢客户端总等待时长，只测试决策优先级。

请先填完下列函数，再把对应开关改为 `True`。

In [27]:
def exercise_a(fn):
    """幂等指纹必须包含模型/模板/生成预算，且不包含瞬时tick。"""
    req=safe_request('k','hello',timeout=12,max_tokens=5)
    first=fn(req)
    require(type(first) is str and len(first)==64,'应该返回SHA256十六进制摘要')
    require(first==fn(req),'同一请求指纹不稳定')
    for alt in [safe_request('k','world',timeout=12,max_tokens=5),
                safe_request('k','hello',version='v0002',timeout=12,max_tokens=5),
                safe_request('k','hello',timeout=15,max_tokens=5),
                safe_request('k','hello',timeout=12,max_tokens=7)]:
        require(fn(alt)!=first,'指纹忽略了语义参数')
    return {'passed':5,'exercise':'fingerprint'}


def exercise_b(fn):
    """事件续传：只返回比游标新的事件，不重复已有ID。"""
    events=[{'id':f'rs0001:{i}','seq':i} for i in range(4)]
    a=fn(events,None);b=fn(events,'rs0001:1')
    require([e['seq'] for e in a]==[0,1,2,3],'无游标必须返回完整记录')
    require([e['seq'] for e in b]==[2,3],'有游标必须排除已收到的事件')
    try:fn(events,'rs0002:1')
    except ValueError:pass
    else:raise LessonError('其他sid不能当续传游标')
    return {'passed':3,'exercise':'replay_after'}


def exercise_c(fn):
    """明确终止/背压/推进的优先级，并拒绝非法参数。"""
    tests=[((5,5,1,3,4),'deadline'),((4,5,3,3,4),'pause'),
           ((4,5,2,3,4),'generate'),((8,10,3,3,4),'pause')]
    for args,expected in tests:
        require(fn(*args)==expected,'tick决策错误 '+str(args))
    try:fn(-1,5,0,3,4)
    except ValueError:pass
    else:raise LessonError('非法当前tick必须拒绝')
    return {'passed':5,'exercise':'tick_decision'}

In [28]:
def fingerprint_ex(request):
    raise NotImplementedError('请独立实现语义指纹')
RUN_EXERCISE_A=False
if RUN_EXERCISE_A:print(exercise_a(fingerprint_ex))

In [29]:
def replay_ex(events,last_event_id):
    raise NotImplementedError('请独立实现事件游标续传')
RUN_EXERCISE_B=False
if RUN_EXERCISE_B:print(exercise_b(replay_ex))

In [30]:
def tick_decision_ex(now,deadline,pending,max_unacked,slow_budget):
    raise NotImplementedError('请独立实现tick优先级')
RUN_EXERCISE_C=False
if RUN_EXERCISE_C:print(exercise_c(tick_decision_ex))

## 12. 练习参考答案

建议先通过自己的练习，再查看下列独立命名的函数；它们**不会覆盖**上面的空白练习。

In [31]:
def answer_a(req):
    require(isinstance(req,StreamRequest),'需要StreamRequest')
    return digest(asdict(req))


def answer_b(events,last_event_id):
    require(type(events) is list and bool(events),'事件非空')
    sid=events[0]['id'].split(':')[0]
    if last_event_id is None:cursor=-1
    else:
        require(last_event_id.startswith(sid+':'),'续传游标跨操作')
        cursor=int(last_event_id.split(':')[1])
    return [e for e in events if e['seq']>cursor]


def answer_c(current_tick,deadline_tick,pending,max_unacked,slow_budget):
    require(all(type(x) is int and x>=0 for x in
                (current_tick,deadline_tick,pending,max_unacked,slow_budget)), 'tick参数非法')
    require(max_unacked>0 and slow_budget>0,'窗口/预算必须正')
    if current_tick>=deadline_tick:return 'deadline'
    if pending>=max_unacked:return 'pause'
    return 'generate'

In [32]:
print('A:',exercise_a(answer_a))
print('B:',exercise_b(answer_b))
print('C:',exercise_c(answer_c))

A: {'passed': 5, 'exercise': 'fingerprint'}
B: {'passed': 3, 'exercise': 'replay_after'}
C: {'passed': 5, 'exercise': 'tick_decision'}


## 13. 保存受信模型、独立重载与报告核对

完整运行会写入新的 `lesson_77_outputs/run_时间戳/`：

| 文件 | 内容 |
|---|---|
| `models.pt` | 两个冻结 LoRA 版本与 tokenizer/配置及权重摘要 |
| `scenario.json` | 执行场景的结构化结果（不含正文/身份盐） |
| `service_config.json` | 本次逻辑窗口与容量配置 |
| `checks.json` | 固定检查名称和计数 |
| `reload.json` | 重新建立模型与场景后逐项核对 |
| `manifest.json` | PyTorch、CPU、范围与明确局限 |

重载只复跑固定场景，**不恢复正在处理的真实会话**。生产服务需要持久化事件账本、原子操作记录、跨节点协调和重启语义。本课没有这些保证。

不要从来源不可信的文件加载权重。

In [33]:
def save_models(path:Path,tok,models,seed:int):
    blob={'version':VERSION,'seed':seed,
        'model_config':asdict(next(iter(models.values())).cfg),'tokenizer':tok.contract(),
        'hashes':{key:state_hash(m.state_dict()) for key,m in models.items()},
        'states':{key:{k:t.detach().cpu().clone() for k,t in m.state_dict().items()}
                  for key,m in models.items()}}
    torch.save(blob,path)


def load_models(path:Path):
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'不是第77课受信模型版本')
    tok,models=build_models(blob['seed'])
    require(blob['model_config']==asdict(next(iter(models.values())).cfg)
            and blob['tokenizer']==tok.contract(),'结构或分词约定不匹配')
    require(set(models)==set(blob['states'])==set(blob['hashes']),'版本集合不匹配')
    for name,m in models.items():
        require(state_hash(blob['states'][name])==blob['hashes'][name],
                '权重摘要不匹配，禁止静默嫁接adapter')
        m.load_state_dict(blob['states'][name],strict=True)
        require(state_hash(m.state_dict())==blob['hashes'][name],'权重恢复不一致')
        m.eval().requires_grad_(False)
    return tok,models,blob['seed']


def reload_run(folder):
    folder=Path(folder)
    tok,models,seed=load_models(folder/'models.pt')
    expected=json.loads((folder/'scenario.json').read_text(encoding='utf-8'))
    actual=run_scenarios(seed,model_bundle=(tok,models))
    require(actual==expected,'干净模型重放与报告不一致')
    return {'exact_report_equal':True,
            'model_hash_verified':True,
            'all_physical_KV_freed':actual['resource']['kv_live_after_flush']==0,
            'scope':'scenario_replay_not_live_session_crash_recovery'}


def fixture_demo(output_root:Path=Path('lesson_77_outputs'),seed=7701):
    torch.set_num_threads(1)
    seed_all(seed)
    path=Path(output_root)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%fZ')
    path.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'seed':seed,'python':platform.python_version(),
              'torch':str(torch.__version__),'device':'cpu','status':'running',
              'limitations':['single_process_synchronous','logical_ticks_not_real_latency',
                'no_HTTP_server_no_broker','no_crash_durable_idempotency','no_thread_safe_atomicity',
                'token_ID_events_not_incremental_UTF8_text','synthetic_random_LM_not_pretrained',
                'real_CPU_KV_not_fused_GPU_paged_attention']}
    write_json(path/'manifest.json',manifest)
    try:
        tok,models=build_models(seed)
        save_models(path/'models.pt',tok,models,seed)
        report=run_scenarios(seed,model_bundle=(tok,models))
        write_json(path/'scenario.json',report)
        write_json(path/'service_config.json',asdict(StreamConfig()))
        write_json(path/'sources.json',SOURCES)
        result=reload_run(path)
        write_json(path/'reload.json',result)
        manifest['status']='completed'
        write_json(path/'manifest.json',manifest)
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(path/'manifest.json',manifest)
        raise
    print('输出目录:',path)
    print('真实KV处理token:',report['resource']['prompt_tokens_processed'],
          '生成token:',report['resource']['generation_tokens'])
    print('断线重试只执行一次:',report['retry_replay']['same_sid'])
    print('终态KV页最终占用:',report['resource']['kv_live_after_flush'])
    return path

In [34]:
RUN_FULL_DEMO=True
DEMO_DIR=None
if RUN_FULL_DEMO:
    DEMO_DIR=fixture_demo(Path('lesson_77_outputs'),7701)
    write_json(DEMO_DIR/'checks.json',checks)
else:
    print('完整报告已显式跳过，数学和检查仍可运行。')

输出目录: lesson_77_outputs/run_20261105_025313_782537Z
真实KV处理token: 45 生成token: 10
断线重试只执行一次: True
终态KV页最终占用: 0


In [35]:
if DEMO_DIR is not None:
    restored=reload_run(DEMO_DIR)
    print(json.dumps(restored,ensure_ascii=False,indent=2))
    print('输出文件:',sorted(p.name for p in DEMO_DIR.iterdir()))

{
  "exact_report_equal": true,
  "model_hash_verified": true,
  "all_physical_KV_freed": true,
  "scope": "scenario_replay_not_live_session_crash_recovery"
}
输出文件: ['checks.json', 'manifest.json', 'models.pt', 'reload.json', 'scenario.json', 'service_config.json', 'sources.json']


## 14. 阶段验收与下一课

完成本课应能解释：

1. 为什么重试不能使用新 deadline、不能重复 prefill，也不能更改 LoRA 版本？
2. 为什么 `Last-Event-ID` + 服务端重放仍需要客户端按 ID 去重？
3. 为什么断线、显式取消、慢消费者关闭、超时和容量拒绝是不同状态？
4. 为什么要先提交 K/V、再发送 token，且终态必须释放物理页？
5. 为什么 TTL 到期后返回“不可重放”，比偷偷重新跑更安全？
6. 为什么单进程幂等**不等于**跨节点、崩溃恢复后的 exactly-once？

参考资料：[MDN SSE](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events)、[HTML SSE 标准](https://html.spec.whatwg.org/multipage/server-sent-events.html)、[AWS 安全重试](https://aws.amazon.com/builders-library/making-retries-safe-with-idempotent-APIs/)、[IETF Idempotency-Key draft](https://datatracker.ietf.org/doc/html/draft-ietf-httpapi-idempotency-key-header-08)、[vLLM前缀缓存](https://docs.vllm.ai/en/latest/design/prefix_caching/)。

**下一课：第 78 课——推理服务的可观测性：Trace、结构化日志、告警与故障定位。** 接着把请求ID、模型身份、资源回收和 SLO 异常串成完整的诊断链。